In [ ]:
# Install dependencies
!pip install flwr[simulation] ucimlrepo torch torchvision matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.7/66.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from ucimlrepo import fetch_ucirepo
import flwr as fl
from flwr.server.strategy import FedAvg
import matplotlib.pyplot as plt
from flwr.common import Metrics
from typing import List, Tuple, Dict, Optional

# 1. Load and preprocess the Statlog Heart dataset
statlog_heart = fetch_ucirepo(id=145)
X = statlog_heart.data.features.values.astype(np.float32)
y = statlog_heart.data.targets.values.astype(np.int64).flatten()
X = (X - X.mean(axis=0)) / X.std(axis=0)
y = (y > 0).astype(np.int64)  # Convert to binary classification

# 2. Partition the data among clients
NUM_CLIENTS = 5
partition_size = len(X) // NUM_CLIENTS
partitions = [
    (X[i*partition_size:(i+1)*partition_size], y[i*partition_size:(i+1)*partition_size])
    for i in range(NUM_CLIENTS)
]

In [ ]:
# 3. Model definition
class HeartModel(nn.Module):
    def __init__(self, model_type="simple_nn"):
        super().__init__()
        self.model_type = model_type
        if model_type == "linear":
            self.layers = nn.Sequential(
                nn.Linear(13, 1),
                nn.Sigmoid()
            )
        elif model_type == "simple_nn":
            self.layers = nn.Sequential(
                nn.Linear(13, 32),
                nn.ReLU(),
                nn.Linear(32, 1),
                nn.Sigmoid()
            )
        elif model_type == "deep_nn":
            self.layers = nn.Sequential(
                nn.Linear(13, 64),
                nn.ReLU(),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, 1),
                nn.Sigmoid()
            )
        elif model_type == "conv_nn":
            self.layers = nn.Sequential(
                nn.Conv1d(1, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Flatten(),
                nn.Linear(16*13, 1),
                nn.Sigmoid()
            )
        elif model_type == "attention_nn":
            self.attention = nn.MultiheadAttention(13, 1)
            self.fc = nn.Linear(13, 1)
            self.sigmoid = nn.Sigmoid()
        else:
            raise ValueError("Invalid model type")

    def forward(self, x):
        if self.model_type == "attention_nn":
            x = x.unsqueeze(0)  # Add sequence dimension
            attn_out, _ = self.attention(x, x, x)
            return self.sigmoid(self.fc(attn_out.squeeze())).squeeze()
        return self.layers(x).squeeze()

In [ ]:
# 4. Flower client
class FlowerClient(fl.client.NumPyClient):
    def __init__(self, model, trainloader, valloader):
        self.model = model
        self.trainloader = trainloader
        self.valloader = valloader
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def get_parameters(self, config=None):
        return [val.cpu().numpy() for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = {k: torch.tensor(v) for k, v in params_dict}
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config=None):
        self.set_parameters(parameters)
        optimizer = optim.Adam(self.model.parameters(), lr=0.01)
        criterion = nn.BCELoss()
        self.model.train()
        for _ in range(3):  # 3 local epochs
            for X_batch, y_batch in self.trainloader:
                X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                optimizer.zero_grad()
                outputs = self.model(X_batch)
                loss = criterion(outputs, y_batch.float())
                loss.backward()
                optimizer.step()
        return self.get_parameters(), len(self.trainloader.dataset), {}

    def evaluate(self, parameters, config=None):
        self.set_parameters(parameters)
        criterion = nn.BCELoss()
        loss, correct = 0.0, 0
        self.model.eval()
        with torch.no_grad():
            for X_batch, y_batch in self.valloader:
                X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                outputs = self.model(X_batch)
                loss += criterion(outputs, y_batch.float()).item() * X_batch.size(0)
                correct += ((outputs > 0.5) == y_batch).sum().item()
        accuracy = correct / len(self.valloader.dataset)
        return loss / len(self.valloader.dataset), len(self.valloader.dataset), {"accuracy": accuracy}

# 5. Define client and server apps
def client_fn(cid: str) -> fl.client.Client:
    idx = int(cid)
    X_part, y_part = partitions[idx]
    tensor_x = torch.tensor(X_part, dtype=torch.float32)
    tensor_y = torch.tensor(y_part, dtype=torch.long)
    dataset = TensorDataset(tensor_x, tensor_y)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    trainset, valset = random_split(dataset, [train_size, val_size])
    trainloader = DataLoader(trainset, batch_size=16, shuffle=True)
    valloader = DataLoader(valset, batch_size=16)

    # Create model - change model_type to test different architectures
    model = HeartModel(model_type="simple_nn")  # Options: linear, simple_nn, deep_nn, conv_nn, attention_nn
    return FlowerClient(model, trainloader, valloader).to_client()

# 6. Define metrics aggregation function
def weighted_average(metrics: List[Tuple[int, Metrics]]) -> Metrics:
    # Multiply accuracy of each client by number of examples used
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]

    # Aggregate and return custom metrics (weighted average)
    return {"accuracy": sum(accuracies) / sum(examples)}

# 7. Define server strategy with metrics aggregation
strategy = FedAvg(
    fraction_fit=1.0,
    fraction_evaluate=1.0,
    min_fit_clients=NUM_CLIENTS,
    min_evaluate_clients=NUM_CLIENTS,
    evaluate_metrics_aggregation_fn=weighted_average,  # Add metrics aggregation
)

# 8. Create server app
server_app = fl.server.ServerApp(
    config=fl.server.ServerConfig(num_rounds=10),
    strategy=strategy,
)

# 9. Create client app
client_app = fl.client.ClientApp(
    client_fn=client_fn,
)

Setting `min_available_clients` lower than `min_fit_clients` or
`min_evaluate_clients` can cause the server to fail when there are too few clients
connected to the server. `min_available_clients` must be set to a value larger
than or equal to the values of `min_fit_clients` and `min_evaluate_clients`.

Setting `min_available_clients` lower than `min_fit_clients` or
`min_evaluate_clients` can cause the server to fail when there are too few clients
connected to the server. `min_available_clients` must be set to a value larger
than or equal to the values of `min_fit_clients` and `min_evaluate_clients`.


            Check the following `FEATURE UPDATE` warning message for the preferred
            new mechanism to use this feature in Flower.
        

            Check the following `FEATURE UPDATE` warning message for the preferred
            new mechanism to use this feature in Flower.
        
            ------------------------------------------------------------
        

        d

In [ ]:
# 10. Run the simulation with history tracking
hist = fl.simulation.run_simulation(
    client_app=client_app,
    server_app=server_app,
    num_supernodes=NUM_CLIENTS,
    backend_config={"client_resources": {"num_cpus": 2}}
)

DEBUG:flwr:Asyncio event loop already running.
INFO :      Starting Flower ServerApp, config: num_rounds=10, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Requesting initial parameters from one random client
(pid=1740) 2025-06-20 15:21:32.814154: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=1740) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(pid=1740) E0000 00:00:1750432892.841256    1740 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
(pid=1740) E0000 00:00:1750432892.849029    1740 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
INFO :      Received initial parameters from one random client
INFO :      Starting

In [ ]:
# 11. Extract and visualize metrics
if hist is not None:
    # Extract accuracy from global metrics
    accuracies = [metrics["accuracy"] for _, metrics in hist.metrics_centralized]

    plt.figure(figsize=(8,5))
    plt.plot(accuracies, marker="o")
    plt.title("Federated Learning Accuracy per Round")
    plt.xlabel("Communication round")
    plt.ylabel("Accuracy")
    plt.grid(True)
    plt.show()

    print(f"\nFinal accuracy: {accuracies[-1]:.4f}")
else:
    print("Simulation completed but no history was returned")


Simulation completed but no history was returned
